# 02 — Streaming Ingestion (Auto Loader → Bronze)

Uses Spark Structured Streaming with **Auto Loader** (`cloudFiles`) to
incrementally ingest the JSON files we generated in notebook 01.

Auto Loader automatically detects new files as they land and processes them
exactly once — no custom bookkeeping, no re-processing.

In [ ]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

VOLUME_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/casino_raw_events"
BRONZE_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_bronze"
CHECKPOINT_PATH = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/casino_raw_events/_checkpoints/bronze"

## Stream from JSON files with Auto Loader

This is the **file-based** approach. Auto Loader watches a directory and
picks up new files incrementally — great for cloud storage landing zones.

In [ ]:
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
    .load(VOLUME_PATH)
)

print("Streaming schema:")
raw_stream.printSchema()

## What if your source is Kafka?

If your data comes from **Kafka** instead of files, the change is minimal.
Here's what the equivalent Kafka read looks like:

```python
# ---- Kafka version (swap in for the cloudFiles read above) ----
# raw_stream = (
#     spark.readStream
#     .format("kafka")
#     .option("kafka.bootstrap.servers", "your-broker:9092")
#     .option("subscribe", "slot_events")
#     .option("startingOffsets", "earliest")
#     .load()
#     .selectExpr("CAST(value AS STRING) as json_str")
#     .select(from_json("json_str", schema).alias("data"))
#     .select("data.*")
# )
```

Everything downstream (Bronze table, Silver transforms, analytics) stays
**exactly the same**. The streaming source is the only thing that changes.

## Write the stream to a Bronze Delta table

In [ ]:
query = (
    raw_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)  # process all available data, then stop
    .toTable(BRONZE_TABLE)
)

query.awaitTermination()
print(f"Stream complete. Data written to {BRONZE_TABLE}")

## Verify Bronze table

In [ ]:
bronze_df = spark.table(BRONZE_TABLE)
print(f"Bronze table row count: {bronze_df.count()}")
bronze_df.show(10, truncate=False)